# Review status: experimental draft — metrics and toxicity heuristics require validation

This notebook was statically reviewed on 2026-10-14 but has not been executed end-to-end. Some comparison tables contain manually entered values, the custom Faster R-CNN evaluator is not yet validated, and toxicity/priority values are heuristic demonstrations rather than scientific toxicology. Verify dataset provenance/license and reproduce metrics with a standard evaluator before reporting results.


In [ ]:
import os
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
!pip install -q ultralytics


In [ ]:
from ultralytics import YOLO


In [ ]:
import os

print(os.listdir("/kaggle/input/ocean-waste-detection"))


In [ ]:
DATASET_PATH = "/kaggle/input/ocean-waste-detection"

with open(f"{DATASET_PATH}/data.yaml", "r") as f:
    print(f.read())

In [ ]:
!pip install -q torchvision albumentations opencv-python


In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F


In [ ]:
DATASET_PATH = "/kaggle/input/ocean-waste-detection"


In [ ]:
CLASS_NAMES = ["__background__", "organic", "recycle", "residue", "reusable"]
NUM_CLASSES = len(CLASS_NAMES)


In [ ]:
class OceanWasteDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        boxes = []
        labels = []

        label_path = os.path.join(self.label_dir, img_name.replace(".jpg", ".txt"))

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    cls, xc, yc, bw, bh = map(float, line.split())
                    x1 = (xc - bw/2) * w
                    y1 = (yc - bh/2) * h
                    x2 = (xc + bw/2) * w
                    y2 = (yc + bh/2) * h
                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        image = F.to_tensor(image)
        return image, target


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = OceanWasteDataset(
    f"{DATASET_PATH}/train/images",
    f"{DATASET_PATH}/train/labels"
)

val_ds = OceanWasteDataset(
    f"{DATASET_PATH}/valid/images",
    f"{DATASET_PATH}/valid/labels"
)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)


In [ ]:
model = fasterrcnn_resnet50_fpn(pretrained=True)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torch.nn.Linear(in_features, NUM_CLASSES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

# Load pretrained model
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)

# Replace the classifier head correctly
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES  # includes background
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
import torch

# Collect trainable parameters
params = [p for p in model.parameters() if p.requires_grad]

# Define optimizer
optimizer = torch.optim.AdamW(params, lr=0.0001)


In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = False


In [ ]:
EPOCHS = 10 # reduce first

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for batch_idx, (images, targets) in enumerate(train_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

        # 🔥 Progress print every 20 batches
        if batch_idx % 20 == 0:
            print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss {losses.item():.4f}")

    print(f"Epoch [{epoch+1}/{EPOCHS}] Total Loss: {epoch_loss:.4f}")


In [ ]:
model.eval()

test_ds = OceanWasteDataset(
    f"{DATASET_PATH}/test/images",
    f"{DATASET_PATH}/test/labels"
)

image, _ = test_ds[0]
output = model([image.to(device)])[0]

output


In [ ]:
toxicity_map = {
    1: 60,   # organic
    2: 35,   # recycle
    3: 75,   # residue
    4: 20    # reusable
}

def severity_level(tox):
    if tox >= 70:
        return "HIGH"
    elif tox >= 50:
        return "MEDIUM"
    else:
        return "LOW"

def priority_level(sev):
    return {"HIGH": 1, "MEDIUM": 2, "LOW": 3}[sev]


In [ ]:
final_results = []

for label in output["labels"].cpu().numpy():
    tox = toxicity_map[label]
    sev = severity_level(tox)
    pr = priority_level(sev)

    final_results.append({
        "class": CLASS_NAMES[label],
        "toxicity": tox,
        "severity": sev,
        "priority": pr
    })

final_results


In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "class": "residue",
    "toxicity": 75,
    "severity": "HIGH",
    "priority": 1
}])

df.to_csv("final_results.csv", index=False)


In [ ]:
!pip install -q pycocotools


In [ ]:
from torchvision.ops import box_iou
import torch

def fast_evaluate(model, dataloader, max_batches=10, iou_threshold=0.5):
    model.eval()
    TP, FP, FN = 0, 0, 0

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(dataloader):
            if batch_idx >= max_batches:
                break

            images = [img.to(device) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                pred_boxes = output["boxes"].cpu()
                gt_boxes = target["boxes"]

                if len(gt_boxes) == 0:
                    FP += len(pred_boxes)
                    continue

                if len(pred_boxes) == 0:
                    FN += len(gt_boxes)
                    continue

                ious = box_iou(pred_boxes, gt_boxes)
                max_ious, _ = ious.max(dim=1)

                TP += (max_ious >= iou_threshold).sum().item()
                FP += (max_ious < iou_threshold).sum().item()
                FN += max(0, gt_boxes.shape[0] - TP)

    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)

    return precision * 100, recall * 100


In [ ]:
precision, recall = fast_evaluate(model, val_loader, max_batches=10)

print(f"Precision: {precision:.2f}%")
print(f"Recall: {recall:.2f}%")


In [ ]:
# Do not estimate mAP by multiplying precision and recall.
# mAP requires integrating precision over recall using a valid detection evaluator.
print("mAP@0.5 is NOT computed by this custom Faster R-CNN evaluator. Use a validated evaluator (for example, COCO-style evaluation) before reporting mAP.")


In [ ]:
import matplotlib.pyplot as plt

loss_history = [
    97.3590, 91.2304, 89.1698, 88.5327, 83.4152,
    83.2079, 80.5167, 79.7247, 78.5963, 77.6450
]

epochs = list(range(1, 11))

plt.figure(figsize=(7,5))
plt.plot(epochs, loss_history, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch")
plt.grid(True)
plt.savefig("training_loss_curve.png", dpi=300)
plt.show()


In [ ]:
import pandas as pd

priority_df = pd.DataFrame({
    "Waste_Type": ["Plastic", "Fishing Net", "Glass"],
    "Toxicity_Score": [0.85, 0.92, 0.30],
    "Priority_Level": ["High", "High", "Low"]
})

priority_df.to_csv("priority_output_table.csv", index=False)

In [ ]:
def zone_analysis(waste_class):
    if waste_class in ["plastic", "residue"]:
        return "Coastal Zone"
    elif waste_class in ["oil_spill", "chemical_waste"]:
        return "Open Water Zone"
    elif waste_class in ["organic"]:
        return "River Mouth Zone"
    else:
        return "Unknown Zone"


def impact_analysis(waste_class):
    rules = {
        "residue": {"toxicity": 75, "severity": "HIGH", "priority": 1,
                    "impact": "Severe threat to marine ecosystem"},
        "plastic": {"toxicity": 60, "severity": "MEDIUM", "priority": 2,
                    "impact": "Long-term environmental pollution"},
        "organic": {"toxicity": 30, "severity": "LOW", "priority": 3,
                    "impact": "Minimal ecological disturbance"},
        "recycle": {"toxicity": 20, "severity": "LOW", "priority": 4,
                    "impact": "Recyclable waste"},
    }
    return rules.get(waste_class, {
        "toxicity": 0,
        "severity": "UNKNOWN",
        "priority": 5,
        "impact": "Unknown impact"
    })


In [ ]:
final_result = {
    "Zone": "Coral Reef Area",
    "Waste_Type": "Plastic Bottle",
    "Detection_Confidence": 0.91,
    "Toxicity_Score": 0.78,
    "Cleanup_Priority": "High"
}

import pandas as pd

zone_impact_df = pd.DataFrame([final_result])
zone_impact_df

In [ ]:
zone_impact_df.to_csv("zone_impact_analysis.csv", index=False)


In [ ]:
CLASS_NAMES = {
    1: "organic",
    2: "recycle",
    3: "residue",
    4: "reusable"
}


In [ ]:
results = []

model.eval()
with torch.no_grad():
    for idx, (images, targets) in enumerate(val_loader):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for i, output in enumerate(outputs):
            if len(output["labels"]) == 0:
                continue

            label_id = int(output["labels"][0])
            waste_class = CLASS_NAMES[label_id]

            zone = zone_analysis(waste_class)
            impact = impact_analysis(waste_class)

            results.append({
                "Image ID": f"image_{idx}_{i}",
                "Zone": zone,
                "Waste Type": waste_class,
                "Toxicity (%)": impact["toxicity"],
                "Severity": impact["severity"],
                "Priority": impact["priority"],
                "Impact": impact["impact"]
            })

        if idx >= 10:   # limit for fast demo
            break


In [ ]:
results = []

model.eval()
with torch.no_grad():
    for idx, (images, targets) in enumerate(val_loader):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for i, output in enumerate(outputs):
            if len(output["labels"]) == 0:
                continue

            # 1️⃣ Detection outputs
            label_id = int(output["labels"][0])
            confidence = float(output["scores"][0])

            waste_class = CLASS_NAMES[label_id]

            # 2️⃣ Zone & impact
            zone = zone_analysis(waste_class)
            impact = impact_analysis(waste_class)
            toxicity = impact["toxicity"]

            # 3️⃣ Risk score (NEW)
            risk_score = (toxicity * 0.6) + (confidence * 100 * 0.4)

            # 4️⃣ Priority adjustment (NEW)
            if risk_score > 70:
                adjusted_priority = 1
            elif risk_score > 40:
                adjusted_priority = 2
            else:
                adjusted_priority = 3

            # 5️⃣ Final record
            results.append({
                "Image ID": f"image_{idx}_{i}",
                "Zone": zone,
                "Waste Type": waste_class,
                "Confidence": round(confidence, 3),
                "Toxicity (%)": toxicity,
                "Risk Score": round(risk_score, 2),
                "Severity": impact["severity"],
                "Priority": adjusted_priority,
                "Impact": impact["impact"]
            })

        if idx >= 5:  # limit for demo
            break


In [ ]:
import pandas as pd

final_df = pd.DataFrame(results)
final_df.to_csv("zone_impact_risk_analysis.csv", index=False)
final_df.head()


In [ ]:
!pip install -q ultralytics

import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = []


In [ ]:
import pandas as pd
import torch

# ---------- Display full text ----------
pd.set_option('display.max_colwidth', None)

# ---------- Class Mapping ----------
CLASS_NAMES = {
    1: "organic",
    2: "recycle",
    3: "residue",
    4: "reusable"
}

# ---------- Zone Analysis ----------
def zone_analysis(waste_class):
    if waste_class in ["residue", "recycle", "reusable"]:
        return "Coastal Zone"
    elif waste_class in ["organic"]:
        return "River Mouth Zone"
    elif waste_class in ["oil_spill", "chemical_waste"]:
        return "Open Water Zone"
    else:
        return "Unknown Zone"

# ---------- Impact + Toxicity ----------
def impact_analysis(waste_class):
    rules = {
        "residue": {
            "toxicity": 75,
            "severity": "HIGH",
            "impact": "Severe threat to marine ecosystem"
        },
        "recycle": {
            "toxicity": 40,
            "severity": "MEDIUM",
            "impact": "Recoverable waste with handling risk"
        },
        "reusable": {
            "toxicity": 25,
            "severity": "LOW",
            "impact": "Low environmental threat"
        },
        "organic": {
            "toxicity": 30,
            "severity": "LOW",
            "impact": "Biodegradable with minor disturbance"
        }
    }
    return rules.get(waste_class, {
        "toxicity": 0,
        "severity": "UNKNOWN",
        "impact": "Unknown environmental impact"
    })

# ---------- Confidence Reliability ----------
def confidence_band(conf):
    if conf >= 0.9:
        return "Very High"
    elif conf >= 0.75:
        return "High"
    elif conf >= 0.6:
        return "Medium"
    else:
        return "Low"

# ---------- Environmental Sensitivity ----------
def sensitivity_index(zone):
    return {
        "Coastal Zone": "High Sensitivity",
        "River Mouth Zone": "Medium Sensitivity",
        "Open Water Zone": "Low Sensitivity",
        "Unknown Zone": "Unknown"
    }.get(zone, "Unknown")

# ---------- Action Recommendation ----------
def action_recommendation(priority):
    if priority == 1:
        return "Immediate cleanup required (24–48 hrs)"
    elif priority == 2:
        return "Scheduled cleanup (within 1 week)"
    else:
        return "Monitor and include in routine cleanup"

# ---------- Cleanup Cost Estimation ----------
def cleanup_cost_estimation(waste_class, zone, risk_score):
    base_cost = {
        "residue": 4000,
        "recycle": 2000,
        "reusable": 1500,
        "organic": 1000
    }.get(waste_class, 1200)

    zone_multiplier = {
        "Coastal Zone": 1.2,
        "River Mouth Zone": 1.1,
        "Open Water Zone": 1.5,
        "Unknown Zone": 1.0
    }.get(zone, 1.0)

    risk_multiplier = 1 + (risk_score / 100)
    return round(base_cost * zone_multiplier * risk_multiplier, 2)

# ---------- MAIN ANALYSIS ----------
results = []

model.eval()
with torch.no_grad():
    for idx, (images, targets) in enumerate(val_loader):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for i, output in enumerate(outputs):
            if len(output["labels"]) == 0:
                continue

            label_id = int(output["labels"][0])
            confidence = float(output["scores"][0])
            waste_class = CLASS_NAMES.get(label_id, "unknown")

            zone = zone_analysis(waste_class)
            impact = impact_analysis(waste_class)
            toxicity = impact["toxicity"]

            # ---------- Risk Score ----------
            risk_score = (toxicity * 0.6) + (confidence * 100 * 0.4)

            # ---------- Priority ----------
            if risk_score > 70:
                priority = 1
            elif risk_score > 40:
                priority = 2
            else:
                priority = 3

            # ---------- Cleanup Cost ----------
            cleanup_cost = cleanup_cost_estimation(waste_class, zone, risk_score)

            # ---------- Explainable Decision ----------
            explanation = (
                f"Waste={waste_class}, Zone={zone}, "
                f"Confidence={confidence:.2f}, Toxicity={toxicity}%, "
                f"RiskScore={risk_score:.1f}, Priority={priority}, "
                f"CleanupCost=₹{cleanup_cost}"
            )

            results.append({
                "Image ID": f"image_{idx}_{i}",
                "Waste Type": waste_class,
                "Zone": zone,
                "Environmental Sensitivity": sensitivity_index(zone),
                "Confidence": round(confidence, 3),
                "Confidence Level": confidence_band(confidence),
                "Toxicity (%)": toxicity,
                "Risk Score": round(risk_score, 2),
                "Severity": impact["severity"],
                "Priority Level": priority,
                "Recommended Action": action_recommendation(priority),
                "Estimated Cleanup Cost (₹)": cleanup_cost,
                "Environmental Impact": impact["impact"],
                "Decision Explanation": explanation
            })

        if idx >= 5:  # demo limit
            break

# ---------- SAVE CSV ----------
final_df = pd.DataFrame(results)
final_df.to_csv("final_environmental_risk_analysis.csv", index=False)

final_df.head()


In [ ]:
data_yaml = """
path: /kaggle/input/ocean-waste-detection
train: train/images
val: val/images

nc: 4
names:
  0: organic
  1: recycle
  2: residue
  3: reusable
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created successfully ✅")


In [ ]:
import os

base = "/kaggle/input/ocean-waste-detection"

for root, dirs, files in os.walk(base):
    print(root)
    break  # only top level


In [ ]:
print("TRAIN:", os.listdir(base + "/train"))
print("VAL:", os.listdir(base))


In [ ]:
import yaml

DATA_YAML_PATH = "/kaggle/input/ocean-waste-detection/data.yaml"

with open(DATA_YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

print(data)


In [ ]:
data_yaml = """
path: /kaggle/input/ocean-waste-detection

train: train/images
val: valid/images
test: test/images

nc: 4
names:
  0: organic
  1: recycle
  2: residue
  3: reusable
"""

with open("data_fixed.yaml", "w") as f:
    f.write(data_yaml)

print("✅ Kaggle-safe data_fixed.yaml created")


In [ ]:
import os

base = "/kaggle/input/ocean-waste-detection"

print("Train images:", len(os.listdir(base + "/train/images")))
print("Valid images:", len(os.listdir(base + "/valid/images")))
print("Test images:", len(os.listdir(base + "/test/images")))


In [ ]:
from ultralytics import YOLO

yolo = YOLO("yolov8n.pt")

yolo.train(
    data="data_fixed.yaml",
    epochs=3,          # keep low for comparison
    imgsz=640,
    batch=16,
    device=0,
    name="yolo_ocean_waste"
)


In [ ]:
results = yolo.val(data="data_fixed.yaml")

print({
    "mAP@0.5": results.box.map50,
    "Precision": results.box.mp,
    "Recall": results.box.mr
})


In [ ]:
import torch, torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = []  # comparison table


In [ ]:
!pip install -q ultralytics
from ultralytics import RTDETR

results = []

rtdetr = RTDETR("rtdetr-l.pt")

rtdetr.train(
    
    data="/kaggle/input/ocean-waste-detection/data.yaml",
    epochs=5,
    imgsz=640,
    batch=8,
    device=0
)

metrics = rtdetr.val()

import numpy as np

results.append({
    "Model": "RT-DETR",
    "Type": "Transformer",
    "Precision (%)": round(float(np.mean(metrics.box.p) * 100), 2),
    "Recall (%)": round(float(np.mean(metrics.box.r) * 100), 2),
    "mAP@0.5 (%)": round(float(np.mean(metrics.box.map50) * 100), 2),
    "mAP@0.5:0.95 (%)": round(float(np.mean(metrics.box.map) * 100), 2)
})


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert list → DataFrame
results_df = pd.DataFrame(results)

metrics = ["Precision (%)", "Recall (%)", "mAP@0.5 (%)"]

results_df.set_index("Model")[metrics].plot(
    kind="bar", figsize=(8,5)
)

plt.title("Model Performance Comparison")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert list → DataFrame
results_df = pd.DataFrame(results)

plt.figure(figsize=(6,4))
plt.bar(
    results_df["Model"],
    results_df["mAP@0.5:0.95 (%)"]
)

plt.title("Localization Quality Comparison (mAP@0.5:0.95)")
plt.ylabel("Percentage (%)")
plt.grid(axis="y")
plt.show()

In [ ]:
import os, torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

CLASS_NAMES = ["organic", "recycle", "residue", "reusable"]

class WasteClassificationDataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = os.listdir(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(
            self.label_dir, img_name.replace(".jpg", ".txt")
        )

        image = Image.open(img_path).convert("RGB")

        with open(label_path) as f:
            class_id = int(f.readline().split()[0])

        if self.transform:
            image = self.transform(image)

        return image, class_id


In [ ]:
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

train_ds = WasteClassificationDataset(
    "/kaggle/input/ocean-waste-detection/train/images",
    "/kaggle/input/ocean-waste-detection/train/labels",
    transform
)

val_ds = WasteClassificationDataset(
    "/kaggle/input/ocean-waste-detection/valid/images",
    "/kaggle/input/ocean-waste-detection/valid/labels",
    transform
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)


In [ ]:
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def train_eval(model, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    model.to(device)

    for _ in range(epochs):
        model.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x,y in val_loader:
            x = x.to(device)
            preds = model(x).argmax(1).cpu()
            y_true.extend(y)
            y_pred.extend(preds)

    return {
        "Accuracy": accuracy_score(y_true, y_pred)*100,
        "F1": f1_score(y_true, y_pred, average="macro")*100,
        "Precision": precision_score(y_true, y_pred, average="macro")*100,
        "Recall": recall_score(y_true, y_pred, average="macro")*100
    }


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
from torchvision.models import vgg19, VGG19_Weights
import torch.nn as nn

vgg = vgg19(weights=VGG19_Weights.DEFAULT)

# Freeze backbone
for param in vgg.features.parameters():
    param.requires_grad = False

# Replace classifier
vgg.classifier[6] = nn.Linear(4096, 4)


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def train_eval(model, epochs=3):
    model.to(device)

    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # ---- TRAIN ----
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}/{epochs} done")

    # ---- EVALUATE ----
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro"
    )

    return {
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(p * 100, 2),
        "Recall (%)": round(r * 100, 2),
        "F1-score (%)": round(f1 * 100, 2)
    }


In [ ]:
vgg_metrics = train_eval(vgg, epochs=3)
vgg_metrics


In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def train_eval(model, epochs=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model.to(device)

    # ---------- TRAIN ----------
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f}")

    # ---------- EVALUATE ----------
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro"
    )

    return {
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(precision * 100, 2),
        "Recall (%)": round(recall * 100, 2),
        "F1-score (%)": round(f1 * 100, 2)
    }


In [ ]:
import os, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

class YOLOClsDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png'))]
        self.tf = T.Compose([
            T.Resize((224,224)),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

    def __len__(self): return len(self.imgs)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.img_dir, self.imgs[i])).convert("RGB")
        with open(os.path.join(self.lbl_dir, self.imgs[i].rsplit('.',1)[0]+".txt")) as f:
            label = int(f.readline().split()[0])
        return self.tf(img), label

train_loader = DataLoader(
    YOLOClsDataset("/kaggle/input/ocean-waste-detection/train/images",
                   "/kaggle/input/ocean-waste-detection/train/labels"),
    batch_size=32, shuffle=True, num_workers=2)

val_loader = DataLoader(
    YOLOClsDataset("/kaggle/input/ocean-waste-detection/valid/images",
                   "/kaggle/input/ocean-waste-detection/valid/labels"),
    batch_size=32, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def train_eval(model, epochs=3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x,y in val_loader:
            out = model(x.to(device)).argmax(1).cpu()
            y_pred.extend(out)
            y_true.extend(y)

    p,r,f,_ = precision_recall_fscore_support(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    return {
        "Accuracy (%)": round(acc*100,2),
        "Precision (%)": round(p*100,2),
        "Recall (%)": round(r*100,2),
        "F1 (%)": round(f*100,2)
    }


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

eff = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# 🔥 Freeze backbone
for p in eff.features.parameters():
    p.requires_grad = False

# Replace classifier
eff.classifier[1] = nn.Linear(1280, 4)

# Train fast
eff_metrics = train_eval(eff, epochs=2)
eff_metrics



In [ ]:
class MHSA(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)

    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        x = x.view(B, C, -1).permute(0, 2, 1)  # (B, HW, C)
        x, _ = self.attn(x, x, x)
        return x.mean(dim=1)


In [ ]:
from torchvision.models import efficientnet_b0

class EffNet_MHSA(nn.Module):
    def __init__(self):
        super().__init__()
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features = base.features
        self.mhsa = MHSA(1280, heads=4)
        self.classifier = nn.Linear(1280, 4)

    def forward(self, x):
        x = self.features(x)      # CNN features
        x = self.mhsa(x)          # Transformer attention
        return self.classifier(x)

# Train MHSA model
eff_mhsa = EffNet_MHSA()
eff_mhsa_metrics = train_eval(eff_mhsa, epochs=3)
eff_mhsa_metrics


In [ ]:
import torch
import torch.nn as nn

class MHSA(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)

    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        x = x.view(B, C, -1).permute(0, 2, 1)  # (B, HW, C)
        x, _ = self.attn(x, x, x)
        return x.mean(dim=1)


In [ ]:
from torchvision.models import vgg19, VGG19_Weights

class VGG19_MHSA(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        base = vgg19(weights=VGG19_Weights.DEFAULT)

        # Use only convolutional layers
        self.features = base.features

        # VGG last feature map = 512 channels
        self.mhsa = MHSA(dim=512, heads=4)

        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.features(x)   # (B, 512, H, W)
        x = self.mhsa(x)       # (B, 512)
        return self.classifier(x)


In [ ]:
vgg_mhsa = VGG19_MHSA()
vgg_mhsa_metrics = train_eval(vgg_mhsa, epochs=3)
vgg_mhsa_metrics


**UNVERIFIED REFERENCE TABLE — manually entered values.** These numbers are not computed from the model evaluation in this cell. Reproduce them from saved checkpoints on a documented held-out test set before citing or presenting them as results.

In [ ]:
import pandas as pd

classification_results = pd.DataFrame([
    {
        "Model": "VGG19",
        "Architecture": "CNN",
        "Accuracy (%)": 94.30,
        "Precision (%)": 96.85,
        "Recall (%)": 94.15,
        "F1-score (%)": 95.31
    },
    {
        "Model": "VGG19 + MHSA",
        "Architecture": "CNN + Multi-Head Self-Attention",
        "Accuracy (%)": 95.80,
        "Precision (%)": 97.54,
        "Recall (%)": 91.44,
        "F1-score (%)": 93.93
    },
    {
        "Model": "EfficientNet-B0",
        "Architecture": "CNN (Compound Scaling)",
        "Accuracy (%)": 73.04,
        "Precision (%)": 78.07,
        "Recall (%)": 56.98,
        "F1-score (%)": 59.45
    },
    {
        "Model": "EfficientNet-B0 + MHSA",
        "Architecture": "CNN + Multi-Head Self-Attention",
        "Accuracy (%)": 98.91,
        "Precision (%)": 98.23,
        "Recall (%)": 99.20,
        "F1-score (%)": 98.69
    }
])

classification_results


In [ ]:
summary_results = pd.DataFrame([
    # Detection models
    ["YOLOv8", "Object Detection", "CNN", 89.83],
    ["Faster R-CNN", "Object Detection", "Two-Stage CNN", 97.01],
    ["RT-DETR-L", "Object Detection", "Transformer", 98.74],

    # Classification models
    ["VGG19", "Classification", "CNN", 94.30],
    ["VGG19 + MHSA", "Classification", "CNN + Attention", 95.80],
    ["EfficientNet-B0", "Classification", "CNN", 73.04],
    ["EfficientNet-B0 + MHSA", "Classification", "CNN + Attention", 98.91]
],
columns=["Model", "Task", "Architecture", "Best Metric (%)"])

summary_results


**UNVERIFIED REFERENCE TABLE — manually entered values.** These detection metrics are not computed from model evaluation in this cell. Re-run the models and use a standard detection evaluator before citing or presenting them as results.

In [ ]:
import pandas as pd

detection_results = pd.DataFrame([
    {
        "Model": "YOLOv8",
        "Architecture": "One-Stage CNN Detector",
        "Precision (%)": 85.84,
        "Recall (%)": 83.66,
        "mAP@0.5 (%)": 89.83,
        "mAP@0.5:0.95 (%)": 70.20
    },
    {
        "Model": "Faster R-CNN",
        "Architecture": "Two-Stage CNN Detector",
        "Precision (%)": 97.01,
        "Recall (%)": 100.00,
        "mAP@0.5 (%)": 97.01,
        "mAP@0.5:0.95 (%)": 82.40
    },
    {
        "Model": "RT-DETR-L",
        "Architecture": "Transformer-based Detector",
        "Precision (%)": 99.28,
        "Recall (%)": 99.24,
        "mAP@0.5 (%)": 98.74,
        "mAP@0.5:0.95 (%)": 88.51
    }
])

detection_results


**UNVERIFIED SUMMARY — manually entered values.** This cell repeats hard-coded classification and detection metrics. It is a display-only draft and must be replaced with values derived from actual evaluation outputs.

In [ ]:
import pandas as pd
import numpy as np

# -------- Classification Results --------
cls_results = pd.DataFrame([
    ["VGG19", "CNN", 94.30, 96.85, 94.15, 95.31],
    ["VGG19 + MHSA", "CNN + MHSA", 95.80, 97.54, 91.44, 93.93],
    ["EfficientNet-B0", "CNN", 73.04, 78.07, 56.98, 59.45],
    ["EfficientNet-B0 + MHSA", "CNN + MHSA", 98.91, 98.23, 99.20, 98.69]
], columns=["Model","Architecture","Accuracy","Precision","Recall","F1"])

# -------- Detection Results --------
det_results = pd.DataFrame([
    ["YOLOv8", "One-stage CNN", 85.84, 83.66, 89.83, 70.20],
    ["Faster R-CNN", "Two-stage CNN", 97.01, 100.00, 97.01, 82.40],
    ["RT-DETR-L", "Transformer", 99.28, 99.24, 98.74, 88.51]
], columns=["Model","Architecture","Precision","Recall","mAP@0.5","mAP@0.5:0.95"])

# Detection F1
det_results["F1"] = 2 * det_results["Precision"] * det_results["Recall"] / \
                    (det_results["Precision"] + det_results["Recall"])

cls_results, det_results


In [ ]:
import matplotlib.pyplot as plt

# ---- Classification Metrics ----
cls_results.set_index("Model")[["Precision","Recall","F1"]].plot(
    kind="bar", figsize=(8,4), title="Classification Performance"
)
plt.ylabel("Score (%)"); plt.grid(axis="y"); plt.show()

# ---- Detection Metrics ----
det_results.set_index("Model")[["Precision","Recall","F1"]].plot(
    kind="bar", figsize=(8,4), title="Detection Precision–Recall–F1"
)
plt.ylabel("Score (%)"); plt.grid(axis="y"); plt.show()

# ---- Detection mAP ----
det_results.set_index("Model")[["mAP@0.5","mAP@0.5:0.95"]].plot(
    kind="bar", figsize=(8,4), title="Detection mAP Comparison"
)
plt.ylabel("mAP (%)"); plt.grid(axis="y"); plt.show()


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.show()

# Example
 #plot_cm(vgg_y_true, vgg_y_pred, "VGG19 Confusion Matrix")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["organic","recycle","residue","reusable"],
        yticklabels=["organic","recycle","residue","reusable"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch

def evaluate_model(model, dataloader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    return y_true, y_pred


In [ ]:
vgg_y_true, vgg_y_pred = evaluate_model(vgg, val_loader)


In [ ]:
plot_cm(vgg_y_true, vgg_y_pred, "VGG19 Confusion Matrix")


In [ ]:
# VGG19 + MHSA
vgg_mhsa_y_true, vgg_mhsa_y_pred = evaluate_model(vgg_mhsa, val_loader)
plot_cm(vgg_mhsa_y_true, vgg_mhsa_y_pred, "VGG19 + MHSA Confusion Matrix")

# EfficientNet-B0
eff_y_true, eff_y_pred = evaluate_model(eff, val_loader)
plot_cm(eff_y_true, eff_y_pred, "EfficientNet-B0 Confusion Matrix")

# EfficientNet-B0 + MHSA
eff_mhsa_y_true, eff_mhsa_y_pred = evaluate_model(eff_mhsa, val_loader)
plot_cm(eff_mhsa_y_true, eff_mhsa_y_pred, "EfficientNet-B0 + MHSA Confusion Matrix")


**UNVERIFIED SUMMARY — manually entered values.** This table repeats hard-coded metrics and combines detection mAP with classification accuracy under a shared column label. Do not interpret the values as a common metric or report them until rebuilt from evaluated results.

In [ ]:
import pandas as pd

combined_results = pd.DataFrame([
    # ---------------- DETECTION MODELS ----------------
    {
        "Task": "Detection",
        "Model": "YOLOv8",
        "Architecture": "One-stage CNN Detector",
        "Accuracy / mAP@0.5 (%)": 89.83,
        "Precision (%)": 85.84,
        "Recall (%)": 83.66,
        "F1-score (%)": None
    },
    {
        "Task": "Detection",
        "Model": "Faster R-CNN",
        "Architecture": "Two-stage CNN Detector",
        "Accuracy / mAP@0.5 (%)": 97.01,
        "Precision (%)": 97.01,
        "Recall (%)": 100.00,
        "F1-score (%)": None
    },
    {
        "Task": "Detection",
        "Model": "RT-DETR-L",
        "Architecture": "Transformer-based Detector",
        "Accuracy / mAP@0.5 (%)": 98.74,
        "Precision (%)": 99.28,
        "Recall (%)": 99.24,
        "F1-score (%)": None
    },

    # ---------------- CLASSIFICATION MODELS ----------------
    {
        "Task": "Classification",
        "Model": "VGG19",
        "Architecture": "CNN",
        "Accuracy / mAP@0.5 (%)": 94.30,
        "Precision (%)": 96.85,
        "Recall (%)": 94.15,
        "F1-score (%)": 95.31
    },
    {
        "Task": "Classification",
        "Model": "VGG19 + MHSA",
        "Architecture": "CNN + Self-Attention",
        "Accuracy / mAP@0.5 (%)": 95.80,
        "Precision (%)": 97.54,
        "Recall (%)": 91.44,
        "F1-score (%)": 93.93
    },
    {
        "Task": "Classification",
        "Model": "EfficientNet-B0",
        "Architecture": "CNN (Compound Scaling)",
        "Accuracy / mAP@0.5 (%)": 73.04,
        "Precision (%)": 78.07,
        "Recall (%)": 56.98,
        "F1-score (%)": 59.45
    },
    {
        "Task": "Classification",
        "Model": "EfficientNet-B0 + MHSA",
        "Architecture": "CNN + Self-Attention",
        "Accuracy / mAP@0.5 (%)": 98.91,
        "Precision (%)": 98.23,
        "Recall (%)": 99.20,
        "F1-score (%)": 98.69
    }
])

combined_results


In [ ]:
import torch

torch.save(model.state_dict(), "best_ocen_model.pth")

In [ ]:
import os
os.listdir()